# Procedimiento ETL — Data Mart PAGMAC (CORREGIDO)

Práctica 4, Bases de Datos Estructuradas 2027-1. Fuente: `pagmac_bk` (MySQL local). Destino: `data_mart_pagmac`.

## Qué se corrigió de tu versión (y en qué celda estaba el error)

1. **Contraseña en texto plano (celdas 1, 22 y 29 de tu versión).** Tu notebook trae la contraseña de root escrita. Este corregido **no** la trae: te la pide con `getpass` y no queda guardada. No subas tu versión anterior a Classroom así: cualquiera que la abra entra a tu MySQL. Cambia esa contraseña.
2. **Equivalencia S/B/MB invertida (tu celda 5).** Tenías `S=10, MB=6`. Es al revés: `MB=10, B=8, S=6` (Muy Bien es lo más alto, Suficiente lo más bajo). Con tu mapa, un MB se promediaba como 6 y un S como 10.
3. **Tu mapa no cubría valores que sí aparecen en tus datos.** Faltaba el `"7"` solo (un 7 quedaba en `NaN` y contaba como reprobado), faltaban decimales como `9.5`, y `NA` quedaba en `NaN` sin avisar. Ahora la conversión es una función: número genérico 0-10, y lo no catalogado va a `errores_etl` con su motivo.
4. **Sexo todo en “Desconocido” (tu celda 8).** Tu salida ya muestra el síntoma: los 5 primeros salen `Desconocido`. Tu mapa solo aceptaba `'1'` y `'2'` exactos. El diagnóstico real (22-nov) muestra `1=273, 2=110, F=3,532, M=7,283`: el par de letras es M/F, así que la `M` es Masculino. La primera versión de este corregido mapeaba la `M` a Mujer por error; ya está corregido a `M` = Hombre, `F` = Mujer, con la palabra completa en `dim_alumno.sexo`.
5. **Periodo invertido (tus celdas 9 y 13).** Tenías `1 = Enero-Mayo` y `2 = Agosto-Diciembre`. Por la regla UNAM del pizarrón (20271 = ago-dic 2026) es al revés: `1 = ago-dic del año anterior`, `2 = ene-jun del año del ciclo`. Ejemplo: `19941` no es “Enero-Mayo 1994”, es ago-dic 1993. Además tu `dim_tiempo` se caía con `TypeError` en `sorted()`: el conjunto mezclaba texto con `NaN` (periodos vacíos), y `is not None` no filtra `NaN`; ya se filtra con `isinstance(p, str)`. Y ojo: `20003` (7,501 filas) era en realidad `20001` (error de captura, confirmado 22-nov: 20001 solo tenía 5 filas) y `01012` (1,450) es el código de extraordinarios sin periodo; ninguno se rechaza en silencio.
6. **Llave de alumno equivocada (tus celdas 10 y 16).** Unías `contenido_calificacion.alumno_id` (= `auth_user.id`) con `contenido_perfilestudiante.id`, y en tu notebook nunca aparece `user_id`. La llave correcta del perfil es `user_id`. Ese error es casi seguro la causa de tus 1,823 “alumnos fantasma” (celda 30): no eran fantasmas, eran alumnos reales cuyo `perfil.id != user_id`. Aquí se diagnostica antes de borrar nada.
7. **Métricas con denominador mal (tu celda 17).** `reprobados` contaba como reprobados a NP, NA y vacíos (todo lo que no era aprobatoria), y `total_alumnos` contaba filas, NP incluidas. Además renombraste el promedio a `porcentaje_aprovechamiento`, pero 7.6 no es un porcentaje. Ahora: evaluados = aprobados + reprobados (NP fuera), reprobación = reprobados/evaluados, aprovechamiento = aprobados/evaluados, promedio = media aparte.
8. **La tabla de hechos no tenía medida ni desduplicación con prioridad (tu celda 20).** Era `drop_duplicates()` de alumno+materia+periodo, que se queda con la primera fila que encuentra, sin regla, y sin la calificación. Los KPIs iban repetidos en cada fila (en tu salida todos los de la materia 1035 valen 48.51 y 7.14). Ahora el hecho lleva `calificacion_num`, `estado_evaluacion`, `acreditado` y `total_intentos`, con prioridad: aprobatoria primero, luego la mayor, luego tipo, luego último `id`.
9. **Extras con umbral inventado (tu celda 19).** `umbral = 2` y las banderas `tiene_mas_extra / tiene_menos_extra` por alumno-materia no responden lo pedido: qué materias tienen más/menos extra. Ahora hay `fact_extra_largo` (1 fila por voto) y el ranking se calcula por materia.
10. **Optativas vacías (tu celda 21).** Filtrabas `tipo_materia == 'OPTATIVA'`, pero los valores reales son `'OP'` / `'OB'`. Por eso `conteo_optativas` salió vacío y subiste una tabla vacía. Ahora el filtro es `'OP'`.
11. **`es_acreditado` y `es_retardo` (tu celda 7).** Tenías acreditado = `> 6`, así que un 6 aprobaba pero no acreditaba, y `es_retardo` (= 6) no es un concepto del pizarrón. Ahora: `acreditado = calificacion >= 6`, y el estado es `APROBADA / REPROBADA / NP`.

**Orden de trabajo:** corre las celdas en orden. El diagnóstico (sección 2) va ANTES de mapear nada. Si ahí aparece un valor nuevo, agrégalo al catálogo antes de seguir.

In [1]:
import re
import os
import getpass
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
import pymysql

# --- Catálogos: cambia SOLO aquí si tu profesor confirma otra equivalencia ---
# SUPUESTO A CONFIRMAR: MB=10, B=8, S=6. AC/CO=7 también es supuesto (condicionado).
EQUIVALENCIA_CALIFICACION = {"MB": 10.0, "B": 8.0, "S": 6.0, "AC": 7.0, "CO": 7.0}
PRIORIDAD_TIPO = {"O": 1, "E": 2, "R": 3, "X": 4}  # Ordinario > Extra > Recurs. > Extra largo

## 1. Conexión (sin contraseña en el código)

Te pide la contraseña al correr la celda y no queda escrita en el notebook. Si la cambiaste (hazlo, tu versión anterior la trae en texto plano), usa la nueva.

In [2]:
USUARIO = "root"
HOST = "127.0.0.1"
PUERTO = "3306"
BASE_FUENTE = "pagmac_bk"
BASE_DM = "data_mart_pagmac"

PASSWORD = getpass.getpass("Contraseña de MySQL (root): ")
engine_src = create_engine(f"mysql+pymysql://{USUARIO}:{PASSWORD}@{HOST}:{PUERTO}/{BASE_FUENTE}")
print("Conexión a la fuente lista.")

Contraseña de MySQL (root):  ········


Conexión a la fuente lista.


In [3]:
# Prueba de conexión + qué tablas hay (tu celda 2, igual)
try:
    print(pd.read_sql_query("SHOW TABLES", engine_src))
except Exception as e:
    print(type(e).__name__, ":", e)

                    Tables_in_pagmac_bk
0                            auth_group
1                auth_group_permissions
2                          auth_message
3                       auth_permission
4                             auth_user
5                      auth_user_groups
6            auth_user_user_permissions
7                     contenido_archivo
8               contenido_archivoevento
9                  contenido_asignatura
10                      contenido_aviso
11               contenido_calificacion
12                    contenido_diahora
13      contenido_equivalencia_materias
14          contenido_estadoimportacion
15                     contenido_evento
16                       contenido_guia
17                    contenido_horario
18               contenido_horario_dias
19               contenido_horarioextra
20            contenido_materiainscrita
21             contenido_perfilegresado
22           contenido_perfilestudiante
23             contenido_perfilprofesor


In [4]:
# Carga: las 6 que ya tenías + auth_user y contenido_semestre, que te faltaban.
# auth_user da el nombre del alumno; contenido_semestre, la descripción del semestre del plan.
# contenido_equivalencia_materias solo se revisa: no se asume qué contiene.
df_calificaciones = pd.read_sql("SELECT * FROM contenido_calificacion", engine_src)
df_asignaturas = pd.read_sql("SELECT * FROM contenido_asignatura", engine_src)
df_perfiles = pd.read_sql("SELECT * FROM contenido_perfilestudiante", engine_src)
df_votos = pd.read_sql("SELECT * FROM contenido_votoextralargo", engine_src)
df_votaciones_ext = pd.read_sql("SELECT * FROM contenido_votacionextraslargos", engine_src)
df_materias_inscritas = pd.read_sql("SELECT * FROM contenido_materiainscrita", engine_src)
df_usuarios = pd.read_sql("SELECT id, username, first_name, last_name, email FROM auth_user", engine_src)
df_semestres = pd.read_sql("SELECT * FROM contenido_semestre", engine_src)
df_equiv_materias = pd.read_sql("SELECT * FROM contenido_equivalencia_materias LIMIT 10", engine_src)

print(f"Calificaciones: {df_calificaciones.shape[0]} filas")
print(f"Perfiles: {df_perfiles.shape[0]} filas | Usuarios: {df_usuarios.shape[0]} filas")
print(f"Votos extra largo: {df_votos.shape[0]} filas")
print("contenido_equivalencia_materias (primeras 10, revisa qué es antes de usarla):")
display(df_equiv_materias)

Calificaciones: 439276 filas
Perfiles: 11198 filas | Usuarios: 11234 filas
Votos extra largo: 2099 filas
contenido_equivalencia_materias (primeras 10, revisa qué es antes de usarla):


,id,E_asignatura_pa_id,E_asignatura_pv,planv
0,29,1044,2602,24021
1,30,1046,2601,24021
2,31,1047,2400,24021
3,32,1054,2303,24021
4,33,1106,2100,24021
5,34,1107,2102,24021
6,35,1108,2104,24021
7,36,1109,2203,24021
8,37,1206,2200,24021
9,38,1206,2300,24021


## 2. Diagnóstico ANTES de mapear (5 minutos, solo lectura)

Tu versión mapeaba sin mirar primero qué valores hay. Estas 6 tablas son el diagnóstico: valor original, valor normalizado y conteo. **Pega estos resultados en el chat** junto con la confirmación de MB/B/S.

In [5]:
def diagnostico(df, columna):
    s = df[columna].astype(str)
    out = (pd.DataFrame({"valor_original": s, "valor_normalizado": s.str.strip().str.upper()})
           .value_counts(dropna=False).reset_index(name="n")
           .sort_values("valor_normalizado").reset_index(drop=True))
    return out

print("Q1. tipo — se supone O/E/R/X")
display(diagnostico(df_calificaciones, "tipo"))
print("Q2. calificacion — fíjate en NA, AC, CO y en si hay decimales")
display(diagnostico(df_calificaciones, "calificacion"))
print("Q3. sexo (perfilestudiante) — en tu versión todo salía Desconocido")
display(diagnostico(df_perfiles, "sexo"))
print("Q4. opt_ob — los valores reales son OP / OB, no OPTATIVA")
display(diagnostico(df_asignaturas, "opt_ob"))
print("Q5. semestre_en_curso")
display(diagnostico(df_calificaciones, "semestre_en_curso"))
print("Q6. periodo (votacionextraslargos)")
display(diagnostico(df_votaciones_ext, "periodo"))

Q1. tipo — se supone O/E/R/X


,valor_original,valor_normalizado,n
0,E,E,92260
1,O,O,347016


Q2. calificacion — fíjate en NA, AC, CO y en si hay decimales


,valor_original,valor_normalizado,n
0,,,727
1,00,00,4
2,05,05,55563
3,06,06,22746
4,07,07,22931
5,08,08,29711
6,09,09,23954
7,10,10,19196
8,13,13,1
9,5,5,1


Q3. sexo (perfilestudiante) — en tu versión todo salía Desconocido


,valor_original,valor_normalizado,n
0,1,1,273
1,2,2,110
2,F,F,3532
3,M,M,7283


... (2 outputs left out)

Q5. semestre_en_curso


,valor_original,valor_normalizado,n
0,01012,01012,1450
1,19581,19581,2
2,19582,19582,3
3,19831,19831,704
4,19832,19832,499
...,...,...,...
58,20101,20101,7980
59,20102,20102,6467
60,20111,20111,7756
61,20112,20112,6803


Q6. periodo (votacionextraslargos)


,valor_original,valor_normalizado,n
0,20102,20102,1
1,20111,20111,1
2,20112,20112,1
3,20121,20121,1
4,20122,20122,1


In [6]:
# Diagnóstico de llaves: la causa probable de tus 1,823 fantasmas.
# En tu versión unías calificacion.alumno_id con perfil.id; la llave es perfil.user_id.
print("perfil.id == perfil.user_id en todas las filas?",
      bool((df_perfiles["id"] == df_perfiles["user_id"]).all()))
print(f"Perfiles donde id != user_id: {(df_perfiles['id'] != df_perfiles['user_id']).sum()} de {len(df_perfiles)}")
print(f"Alumnos en calificaciones sin usuario en auth_user: "
      f"{(~df_calificaciones['alumno_id'].isin(set(df_usuarios['id']))).sum()}")
print(f"Alumnos en calificaciones sin perfil (por user_id): "
      f"{(~df_calificaciones['alumno_id'].isin(set(df_perfiles['user_id']))).sum()}")
print(f"Alumnos en calificaciones sin perfil (por perfil.id, tu join anterior): "
      f"{(~df_calificaciones['alumno_id'].isin(set(df_perfiles['id']))).sum()}")
print("Si el último número es mucho mayor que el anterior, el join anterior era el problema.")

perfil.id == perfil.user_id en todas las filas? False
Perfiles donde id != user_id: 11198 de 11198
Alumnos en calificaciones sin usuario en auth_user: 0
Alumnos en calificaciones sin perfil (por user_id): 0
Alumnos en calificaciones sin perfil (por perfil.id, tu join anterior): 2003
Si el último número es mucho mayor que el anterior, el join anterior era el problema.


## 2b. Investigación del periodo `01012` (1,450 filas, código desconocido)

El número de cuenta es `auth_user.username` (y `auth_user.id` es la llave `alumno_id`). Estas tres salidas dicen por la generación y por los demás periodos de esos mismos alumnos en qué fecha cae `01012`: si son generación 2001 y sus otros periodos son 20011/20012, es 2001-2 con un código raro; si son de otra generación, es otra cosa.

In [7]:
_ids_01012 = df_calificaciones.loc[
    df_calificaciones["semestre_en_curso"].astype(str).str.strip() == "01012", "alumno_id"].unique()
print(f"Alumnos con 01012: {len(_ids_01012)}")

# 1) Quiénes son: número de cuenta, nombre y generación
alumnos_01012 = (
    df_usuarios[df_usuarios["id"].isin(_ids_01012)]
    .merge(df_perfiles.rename(columns={"id": "id_perfil"}),
           left_on="id", right_on="user_id", how="left")
    [["id", "username", "first_name", "last_name", "generacion"]]
    .rename(columns={"id": "alumno_id", "username": "numero_cuenta"})
    .sort_values(["generacion", "alumno_id"])
)
print("Primeros 20 (número de cuenta, nombre, generación):")
display(alumnos_01012.head(20))

# 2) Generación de esos alumnos: si se concentran en una, ahí está la fecha
print("Alumnos con 01012 por generación:")
display(alumnos_01012.groupby("generacion").size().rename("alumnos").reset_index())

# 3) Los demás periodos de esos mismos alumnos, ordenados: dónde encaja 01012
print("Otros periodos de esos alumnos (¿entre 20011 y 20012? ¿antes de 20001?):")
display(df_calificaciones[df_calificaciones["alumno_id"].isin(_ids_01012)]
        .groupby("semestre_en_curso").size().rename("registros")
        .reset_index().sort_values("semestre_en_curso"))

# 4) ¿De qué materias son los registros 01012? Si 1-2 materias concentran
# los 1,450, es un código de materia (taller, idioma, servicio social...);
# si se reparte en todas, es un placeholder de "sin periodo".
_df01012 = df_calificaciones[df_calificaciones["semestre_en_curso"].astype(str).str.strip() == "01012"]
print("Registros 01012 por asignatura (top 10):")
display(_df01012.groupby("asignatura_id").size().rename("registros")
        .reset_index().sort_values("registros", ascending=False).head(10))
print("Registros 01012 por tipo y calificación:")
display(_df01012.groupby(["tipo", "calificacion"]).size().rename("registros").reset_index())

Alumnos con 01012: 699
Primeros 20 (número de cuenta, nombre, generación):


,alumno_id,numero_cuenta,first_name,last_name,generacion
0,599,082201766,John,Doe,1985
1,751,083022665,John,Doe,1986
4,1752,085289277,John,Doe,1989
5,1782,085340729,John,Doe,1989
9,3691,089563722,John,Doe,1989
10,3842,089604326,John,Doe,1989
6,2226,086336886,John,Doe,1990
3,1587,085067204,John,Doe,1991
7,2457,087074642,John,Doe,1991
8,3249,088411778,John,Doe,1991


Alumnos con 01012 por generación:


,generacion,alumnos
0,1985,1
1,1986,1
2,1989,4
3,1990,1
4,1991,3
5,1994,2
6,1995,1
7,1996,2
8,1998,2
9,1999,4


Otros periodos de esos alumnos (¿entre 20011 y 20012? ¿antes de 20001?):


,semestre_en_curso,registros
0,01012,1450
1,19851,10
2,19852,8
3,19861,13
4,19862,19
5,19871,14
6,19872,17
7,19881,20
8,19882,12
9,19891,31


Registros 01012 por asignatura (top 10):


,asignatura_id,registros
12,1106,193
13,1107,152
16,1206,95
14,1108,84
19,1209,69
15,1110,42
63,2601,41
23,1409,32
29,1510,31
65,2604,31


Registros 01012 por tipo y calificación:


,tipo,calificacion,registros
0,E,05,620
1,E,06,143
2,E,07,104
3,E,08,59
4,E,09,28
5,E,10,28
6,E,NP,468


## Conclusión del `01012` (22-nov)

Misterio resuelto: **los 1,450 registros son todos tipo E (extraordinario)**. No es un semestre de calendario ni una materia: es el código que el sistema pone a los **exámenes extraordinarios sin periodo capturado**.

* Se reparte en muchas materias (top 10 = 53%), con 1106 (193) y 1107 (152) a la cabeza; vale la pena ver sus nombres en `contenido_asignatura`.
* Calificaciones: `05` = 620 (43%), `NP` = 468 (32%), aprobados solo 362 (25%). Reprobación entre evaluados: 63%.
* Consecuencia para el mart: estos registros **sí entran al hecho** (son evaluaciones), con su periodo especial `01012`, sin fusionarse con 20012 ni con ningún semestre. En los KPIs por periodo aparecen aparte; en los KPIs por materia/generación sí cuentan.

## 3. Limpieza: las reglas del ETL, en funciones

* **Periodo UNAM (corregido):** `1 = ago-dic del año anterior`, `2 = ene-jun del año del ciclo`. `20271` = ago-dic 2026.
* **Calificación:** número genérico 0-10 (cubre `"7"`, `"9.5"`, `"05"`); texto por catálogo; `NP` → `NaN`; `NA` = no aprobó (válido, sin número, entra al hecho como reprobada y cuenta en los índices); vacío = nulo / sin calificación (fuera del hecho y de los KPIs); texto desconocido → inválido, a `errores_etl`.
* **Sexo (corregido con el diagnóstico):** `1`/`H`/`M` (letra) = Hombre; `2`/`F` = Mujer. La `M` de letra es Masculino: lo prueba que exista `F` (3,532) como su par. Lo que no reconozca queda en `None` y se reporta, no en “Desconocido” silencioso.

In [8]:
def norm_txt(x):
    return "" if pd.isna(x) else str(x).strip().upper()

def normalizar_periodo(x):
    # .map() pasa los NaN sin llamar a la función y pd.concat los deja como
    # float NaN: por eso se filtra con isinstance(p, str) al armar dim_tiempo.
    if pd.isna(x):
        return None
    p = re.sub(r"[-/ ]", "", str(x).strip())
    # 20003 es en realidad 20001 (error de captura, confirmado 22-nov:
    # 20001 solo tenía 5 filas y no hubo paro en esas fechas que
    # justificara un periodo 3; los 7,501 de 20003 son los de 2000-1).
    if p == "20003":
        return "20001"
    return p if re.fullmatch(r"\d{4}[12]", p) else None

def periodo_descripcion(p):
    if p == "01012":
        # Investigación 22-nov (699 alumnos): 97% generación 2000+ (pico
        # 2010), ~2 registros por alumno, y esos alumnos tienen registros en
        # todos los periodos 1985-2012. NO es un semestre de calendario:
        # si fuera 2001-2, sus alumnos serían gen ~1998-2001 y su actividad
        # se concentraría en 2001. Es un código especial (placeholder o
        # categoría sin periodo. Investigación 22-nov: los 1,450 registros
    # son TODOS tipo E (extraordinario), con 05=620 y NP=468. Son
    # extraordinarios sin periodo capturado, no un semestre.
        return "Periodo especial 01012: exámenes extraordinarios sin periodo asignado"
    # 20003 ya no llega aquí: normalizar_periodo lo mapea a 20001.
    anio = int(p[:4])
    if p[4] == "1":
        return f"{anio}-1 (ago-dic {anio - 1})"
    if p[4] == "2":
        return f"{anio}-2 (ene-jun {anio})"
    return f"{anio}-{p[4]} (periodo no estándar, confirmar qué es)"

def es_periodo_estandar(p):
    return p[4] in ("1", "2") and 1900 <= int(p[:4]) <= 2030

def calificacion_a_numero(x):
    # Devuelve (calificacion_num, es_np, es_valida, motivo)
    s = norm_txt(x)
    if s == "NP":
        return (np.nan, True, True, None)
    if s in EQUIVALENCIA_CALIFICACION:
        return (EQUIVALENCIA_CALIFICACION[s], False, True, None)
    if s in ("NA", "", "NAN", "NONE"):
        # NA = "no aprobó" (confirmado 22-nov): válido, sin número, pero SÍ
        # es evaluación: entra al hecho como reprobada (es_na) y cuenta en
        # los índices de reprobación. Los nulos de verdad (vacío / NaN) son
        # los que se marcan es_sin_calificacion y se excluyen del hecho.
        return (np.nan, False, True, None)
    if re.fullmatch(r"\d+(\.\d+)?", s):
        v = float(s)
        if 0 <= v <= 10:
            return (v, False, True, None)
        return (np.nan, False, False, "Calificación numérica fuera de 0-10")
    return (np.nan, False, False, "Calificación desconocida (no está en el catálogo ni es numérica)")

def sexo_a_hm(x):
    # CORRECCIÓN 2026-11-22 (diagnóstico real): los valores son 1, 2, F, M.
    # La F prueba que el par de letras es M/F (Masculino/Femenino), NO H/M:
    # la M de letra es Masculino, no Mujer. La versión anterior mapeaba la M
    # a Mujer, lo cual invertía el sexo de 7,283 perfiles. Se devuelve la
    # palabra completa para que la M ya no se confunda con Mujer.
    if pd.isna(x):
        return None
    s = norm_txt(x).removesuffix(".0")
    if s in ("1", "H", "HOMBRE", "MASCULINO", "M"):
        return "Hombre"
    if s in ("2", "F", "MUJER", "FEMENINO"):
        return "Mujer"
    return None

def opt_ob_norm(x):
    s = norm_txt(x)
    if s in ("OP", "OPT", "OPTATIVA"):
        return "OP"
    if s in ("OB", "OBL", "OBLIGATORIA"):
        return "OB"
    return None

## 4. Dimensiones (corregidas)

* `dim_alumno`: llave `alumno_id = perfil.user_id` (no `perfil.id`), con nombre desde `auth_user`, y LEFT JOIN con el perfil: un alumno sin perfil entra igual (con generación/sexo en nulo). Sin `password`, calle, colonia, teléfono ni flags de staff.
* `dim_asignatura`: `opt_ob` normalizado a `OB/OP` + `es_optativa`, con la descripción del semestre del plan.
* `dim_tiempo` (antes `dim_tiempo`, ahora con nombres claros): `periodo_unam`, `anio_ciclo`, `semestre_ciclo`, descripción corregida.

In [9]:
# CORRECCIÓN del error TypeError en sorted(): el conjunto mezclaba str con
# float NaN (periodos vacíos que .map() pasa sin llamar a la función, y
# `p is not None` NO filtra NaN). Se filtra con isinstance(p, str).
dim_tiempo = pd.DataFrame([{
    "id_periodo": i + 1,
    "periodo_unam": p,
    # 01012 no es un año: anio/semestre quedan en None (antes salía
    # anio_ciclo=101, semestre_ciclo=2, un año absurdo en dim_tiempo).
    "anio_ciclo": None if p == "01012" else int(p[:4]),
    "semestre_ciclo": None if p == "01012" else int(p[4]),
    "descripcion": periodo_descripcion(p),
    "es_periodo_estandar": es_periodo_estandar(p),
} for i, p in enumerate(sorted({
    p for p in pd.concat([
        df_calificaciones["semestre_en_curso"].map(normalizar_periodo),
        df_votaciones_ext["periodo"].map(normalizar_periodo),
    ]) if isinstance(p, str) and p
}))])

dim_asignatura = (
    df_asignaturas
    .merge(df_semestres, left_on="semestre_id", right_on="num", how="left")
    .assign(id_asignatura=lambda d: range(1, len(d) + 1),
            opt_ob=lambda d: d["opt_ob"].map(opt_ob_norm),
            es_optativa=lambda d: (d["opt_ob"] == "OP").astype(int))
    [["id_asignatura", "clave", "nombre", "plan", "semestre_id", "desc", "opt_ob", "es_optativa"]]
    .rename(columns={"semestre_id": "semestre_plan", "desc": "semestre_desc"})
)

dim_alumno = (
    df_usuarios
    # LEFT JOIN: un alumno con calificaciones pero sin perfil sigue entrando
    # a la dimensión (con generación/sexo en nulo); solo se excluye si no
    # existe en auth_user. El id del perfil se renombra antes para no
    # chocar con el id del usuario.
    .merge(df_perfiles.rename(columns={"id": "id_perfil"}),
           left_on="id", right_on="user_id", how="left")
    .assign(id_estudiante=lambda d: range(1, len(d) + 1),
            nombre_completo=lambda d: (d["first_name"].astype(str).str.strip() + " "
                                       + d["last_name"].astype(str).str.strip()).str.strip(),
            generacion=lambda d: pd.to_numeric(d["generacion"], errors="coerce").astype("Int64"),
            sexo=lambda d: d["sexo"].map(sexo_a_hm))
    [["id_estudiante", "id", "username", "nombre_completo", "email",
      "generacion", "sexo", "deleg_muni", "fecha_nac"]]
    .rename(columns={"id": "id_auth_user"})
)
print("dim_tiempo (primeras filas):"); display(dim_tiempo.head())
print("dim_asignatura:"); display(dim_asignatura.head())
print("dim_alumno:"); display(dim_alumno.head())
print(f"Sexo sin reconocer (None) en dim_alumno: {dim_alumno['sexo'].isna().sum()} de {len(dim_alumno)}")
print(f"Perfiles sin usuario en auth_user (no entran a dim_alumno): "
      f"{(~df_perfiles['user_id'].isin(set(df_usuarios['id']))).sum()}")

dim_tiempo (primeras filas):


,id_periodo,periodo_unam,anio_ciclo,semestre_ciclo,descripcion,es_periodo_estandar
0,1,01012,NaN,NaN,Periodo especial 01012: exámenes extraordinari...,False
1,2,19581,1958.0,1.0,1958-1 (ago-dic 1957),True
2,3,19582,1958.0,2.0,1958-2 (ene-jun 1958),True
3,4,19831,1983.0,1.0,1983-1 (ago-dic 1982),True
4,5,19832,1983.0,2.0,1983-2 (ene-jun 1983),True


dim_asignatura:


,id_asignatura,clave,nombre,plan,semestre_plan,semestre_desc,opt_ob,es_optativa
0,1,1000,DESARROLLO Y PLANIF I,24021,9,Noveno,OP,1
1,2,1001,MET Y TEC DE INV CIENC SOC.,24021,9,Noveno,OP,1
2,3,1002,PROC ADMON. PUBL,24021,9,Noveno,OP,1
3,4,1003,ADMON. DE EMPRE EN MEX.,24021,9,Noveno,OP,1
4,5,1004,CUENTAS. NACIO Y BALAN. DE PAG,24021,9,Noveno,OP,1


dim_alumno:


,id_estudiante,id_auth_user,username,nombre_completo,email,generacion,sexo,deleg_muni,fecha_nac
0,1,1,marcos,John Doe,john@doe.com,<NA>,NaN,NaN,NaN
1,2,2,067723193,John Doe,john@doe.com,1985,Hombre,,1961-11-01
2,3,3,069007596,John Doe,john@doe.com,1986,Hombre,,1954-10-28
3,4,4,070078848,John Doe,john@doe.com,1984,Hombre,,1952-07-17
4,5,5,070149726,John Doe,john@doe.com,1986,Hombre,,1954-03-04


Sexo sin reconocer (None) en dim_alumno: 36 de 11234
Perfiles sin usuario en auth_user (no entran a dim_alumno): 0


## 5. Staging de calificaciones

Lo inválido no se borra en silencio: va a `errores_etl` con su motivo. `NA` (47,679 filas en tus datos) ya NO va ahí: es "no aprobó", entra al hecho como reprobada y cuenta en los índices de reprobación (sin número, fuera de promedios). Los vacíos/nulos sí se excluyen del hecho y se cuentan aparte arriba.

In [10]:
stg_calificacion = df_calificaciones.copy()
stg_calificacion["periodo_norm"] = stg_calificacion["semestre_en_curso"].map(normalizar_periodo)
stg_calificacion["calif_norm"] = stg_calificacion["calificacion"].map(norm_txt)
stg_calificacion[["calificacion_num", "es_np", "es_valida", "motivo_rechazo"]] = (
    stg_calificacion["calificacion"].map(calificacion_a_numero).apply(pd.Series))
stg_calificacion.loc[stg_calificacion["periodo_norm"].isna(), ["es_valida", "motivo_rechazo"]] = (
    False, "Periodo inválido (se esperaba AAAAS, ej. 20271)")
stg_calificacion["tipo_norm"] = stg_calificacion["tipo"].map(norm_txt)
stg_calificacion["es_na"] = (stg_calificacion["calif_norm"] == "NA")
stg_calificacion["es_sin_calificacion"] = stg_calificacion["calif_norm"].isin(["", "NAN", "NONE"])
print(f"NA (no aprobó, reprobada): {int(stg_calificacion['es_na'].sum())} filas")
print(f"Sin calificación (vacío / nulo): {int(stg_calificacion['es_sin_calificacion'].sum())} filas")

errores_etl = stg_calificacion.loc[~stg_calificacion["es_valida"],
    ["id", "alumno_id", "asignatura_id", "calificacion", "motivo_rechazo"]].copy()
errores_etl.insert(0, "tabla_origen", "contenido_calificacion")
errores_etl = errores_etl.rename(columns={"id": "id_origen", "calificacion": "dato"})
print(f"staging: {len(stg_calificacion)} filas, válidas: {int(stg_calificacion['es_valida'].sum())}, "
      f"rechazadas: {len(errores_etl)}")
print("Rechazadas por motivo:"); display(errores_etl["motivo_rechazo"].value_counts())
print("Rechazadas por dato:"); display(errores_etl["dato"].astype(str).str.upper().value_counts().head(15))

NA (no aprobó, reprobada): 47679 filas
Sin calificación (vacío / nulo): 727 filas
staging: 439276 filas, válidas: 439275, rechazadas: 1
Rechazadas por motivo:


motivo_rechazo
Calificación numérica fuera de 0-10    1
Name: count, dtype: int64

Rechazadas por dato:


dato
13    1
Name: count, dtype: int64

## 6. Hecho `fact_calificacion` — grano y prioridad

**Grano: 1 fila por alumno + asignatura + periodo**, con la calificación seleccionada. Prioridad (la del pizarrón): 1) aprobatoria (≥ 6), 2) calificación mayor (NP al final), 3) tipo O > E > R > X, 4) último `id`. `total_intentos` guarda cuántas tenía antes de priorizar.

In [11]:
_n_sin_calif = int(stg_calificacion["es_sin_calificacion"].sum())
print(f"Se excluyen del hecho {_n_sin_calif} filas sin calificación (NA / vacío, nulo).")
stg_ok = stg_calificacion[(stg_calificacion["es_valida"])
                          & (~stg_calificacion["es_sin_calificacion"])].copy()
stg_ok["total_intentos"] = (stg_ok.groupby(["alumno_id", "asignatura_id", "periodo_norm"])["id"]
                            .transform("count"))
stg_ok["es_aprobatoria"] = (stg_ok["calificacion_num"] >= 6).fillna(False)
stg_ok["prioridad_tipo"] = stg_ok["tipo_norm"].map(PRIORIDAD_TIPO).fillna(99)

# Alumnos sin dimensión (sin usuario o sin perfil): no se borran en silencio, van a errores_etl
_sin_dim = stg_ok[~stg_ok["alumno_id"].isin(set(dim_alumno["id_auth_user"]))]
if len(_sin_dim):
    _err = _sin_dim[["id", "alumno_id", "asignatura_id", "calificacion"]].copy()
    _err.insert(0, "tabla_origen", "contenido_calificacion")
    _err["motivo_rechazo"] = "Alumno sin usuario/perfil en las dimensiones"
    _err = _err.rename(columns={"id": "id_origen", "calificacion": "dato"})
    errores_etl = pd.concat([errores_etl, _err], ignore_index=True)
    print(f"Calificaciones de alumnos sin dimensión (a errores_etl): {len(_sin_dim)}")
stg_ok = stg_ok[stg_ok["alumno_id"].isin(set(dim_alumno["id_auth_user"]))]

fact_calificacion = (
    stg_ok.sort_values(by=["es_aprobatoria", "calificacion_num", "prioridad_tipo", "id"],
                       ascending=[False, False, True, False], na_position="last")
    .drop_duplicates(subset=["alumno_id", "asignatura_id", "periodo_norm"], keep="first")
    .merge(dim_alumno[["id_estudiante", "id_auth_user"]],
           left_on="alumno_id", right_on="id_auth_user", how="inner")
    .merge(dim_asignatura[["id_asignatura", "clave"]],
           left_on="asignatura_id", right_on="clave", how="inner")
    .merge(dim_tiempo[["id_periodo", "periodo_unam"]],
           left_on="periodo_norm", right_on="periodo_unam", how="inner")
    .assign(
        id_hecho=lambda d: range(1, len(d) + 1),
        es_reprobada=lambda d: (((d["calificacion_num"] < 6) & d["calificacion_num"].notna()) | d["es_na"]),
        estado_evaluacion=lambda d: np.where(d["es_np"], "NP",
            np.where(d["calificacion_num"] >= 6, "APROBADA", "REPROBADA")),
        acreditado=lambda d: (d["calificacion_num"] >= 6).fillna(False).astype(int),
    )
    [["id_hecho", "id_estudiante", "id_asignatura", "id_periodo",
      "calificacion", "calificacion_num", "es_aprobatoria", "es_reprobada",
      "es_np", "es_na", "estado_evaluacion", "acreditado", "tipo_norm", "total_intentos"]]
    .rename(columns={"id_estudiante": "id_dim_estudiante", "id_asignatura": "id_dim_asignatura",
                     "id_periodo": "id_dim_periodo", "calificacion": "calificacion_original",
                     "tipo_norm": "tipo_evaluacion"})
)
print(f"De {len(df_calificaciones)} calificaciones a {len(fact_calificacion)} filas de hecho "
      f"(grano alumno+materia+periodo).")
print(f"errores_etl total: {len(errores_etl)}")
display(fact_calificacion.head())

Se excluyen del hecho 727 filas sin calificación (NA / vacío, nulo).
De 439276 calificaciones a 405157 filas de hecho (grano alumno+materia+periodo).
errores_etl total: 1


,id_hecho,id_dim_estudiante,id_dim_asignatura,id_dim_periodo,calificacion_original,calificacion_num,es_aprobatoria,es_reprobada,es_np,es_na,estado_evaluacion,acreditado,tipo_evaluacion,total_intentos
0,1,10748,107,62,10,10.0,True,False,False,False,APROBADA,1,O,1
1,2,10774,107,62,10,10.0,True,False,False,False,APROBADA,1,O,1
2,3,10757,106,62,10,10.0,True,False,False,False,APROBADA,1,O,1
3,4,10748,106,62,10,10.0,True,False,False,False,APROBADA,1,O,1
4,5,9941,31,62,10,10.0,True,False,False,False,APROBADA,1,O,1


## 7. Métricas (reportes, NO columnas del hecho)

Denominador explícito: evaluados = aprobados + reprobados. **NP fuera** de promedios e índices. `porcentaje_aprovechamiento` ahora sí es un porcentaje (antes era el promedio 0-10 con nombre de porcentaje).

In [12]:
_f = fact_calificacion.merge(dim_tiempo, left_on="id_dim_periodo", right_on="id_periodo")
_f = _f.merge(dim_asignatura, left_on="id_dim_asignatura", right_on="id_asignatura")
_f = _f.merge(dim_alumno[["id_estudiante", "generacion"]],
              left_on="id_dim_estudiante", right_on="id_estudiante")

_sin_np = _f[~_f["es_np"]]
metricas_materia = (
    _sin_np.groupby(["periodo_unam", "clave", "nombre"], as_index=False)
    .agg(evaluados_sin_np=("calificacion_num", "size"),
         aprobados=("es_aprobatoria", "sum"),
         reprobados=("es_reprobada", "sum"),
         promedio_materia=("calificacion_num", "mean"))
    .assign(indice_reprobacion=lambda d: (100 * d["reprobados"] / d["evaluados_sin_np"]).round(2),
            porcentaje_aprovechamiento=lambda d: (100 * d["aprobados"] / d["evaluados_sin_np"]).round(2),
            promedio_materia=lambda d: d["promedio_materia"].round(2))
)
promedio_generacion = (
    _f[_f["calificacion_num"].notna()]
    .groupby(["periodo_unam", "clave", "nombre", "generacion"], as_index=False)
    .agg(n_evaluados_sin_np=("calificacion_num", "size"),
         promedio_materia_generacion=("calificacion_num", "mean"))
    .assign(promedio_materia_generacion=lambda d: d["promedio_materia_generacion"].round(2))
)
print("Métricas por materia y periodo:"); display(metricas_materia.head())
print("Promedio por materia y generación:"); display(promedio_generacion.head())

Métricas por materia y periodo:


,periodo_unam,clave,nombre,evaluados_sin_np,aprobados,reprobados,promedio_materia,indice_reprobacion,porcentaje_aprovechamiento
0,01012,1038,ANALISIS DE INVERSIONES,5,5,0,8.60,0.0,100.0
1,01012,1040,INGENIERIA D SOFTWARE ORIENT.OBJETOS,3,3,0,7.33,0.0,100.0
2,01012,1044,SIMULACION ESTOCASTICA,2,2,0,7.00,0.0,100.0
3,01012,1045,SISTEMAS OPERATIVOS,11,11,0,10.00,0.0,100.0
4,01012,1046,TEORIA DE LA COMPUTACION,5,4,1,6.80,20.0,80.0


Promedio por materia y generación:


,periodo_unam,clave,nombre,generacion,n_evaluados_sin_np,promedio_materia_generacion
0,01012,1038,ANALISIS DE INVERSIONES,2007,5,8.6
1,01012,1040,INGENIERIA D SOFTWARE ORIENT.OBJETOS,2006,1,8.0
2,01012,1040,INGENIERIA D SOFTWARE ORIENT.OBJETOS,2007,2,7.0
3,01012,1044,SIMULACION ESTOCASTICA,2005,1,8.0
4,01012,1044,SIMULACION ESTOCASTICA,2006,1,6.0


## 8. Extras largos y optativas (corregidos)

* Extras: `fact_extra_largo` es 1 fila por voto; el ranking “materias con más/menos extra” se calcula **por materia**, sin umbral inventado.
* Optativas: el filtro es `opt_ob == 'OP'` (tu `'OPTATIVA'` dejaba la tabla vacía). `fact_inscripcion` es 1 fila por alumno + asignatura; como `contenido_materiainscrita` no trae periodo, ese ranking es histórico total.

In [13]:
# dim_votacion (faltaba): la ronda de votación extra larga.
# En el hecho solo estaba id_votacion como número, sin dimensión:
# aquí quedan su periodo y sus fechas de inicio/fin.
dim_votacion = (
    df_votaciones_ext[["id", "periodo", "inicio", "fin"]]
    .rename(columns={"id": "id_votacion", "periodo": "periodo_unam"})
    .drop_duplicates(subset=["id_votacion"])
)
print(f"dim_votacion: {len(dim_votacion)} rondas.")
display(dim_votacion)

dim_votacion: 5 rondas.


,id_votacion,periodo_unam,inicio,fin
0,3,20102,2010-01-05,2010-01-20
1,4,20111,2010-06-29,2010-07-28
2,5,20112,2011-01-06,2011-01-08
3,6,20121,2011-07-27,2011-08-03
4,7,20122,2012-01-16,2012-01-19


In [14]:
fact_extra_largo = (
    df_votos
    .merge(df_votaciones_ext[["id", "periodo"]],
           left_on="votacion_id", right_on="id", how="inner", suffixes=("", "_vot"))
    .assign(periodo_norm=lambda d: d["periodo"].map(normalizar_periodo))
    .merge(dim_alumno[["id_estudiante", "id_auth_user"]],
           left_on="alumno_id", right_on="id_auth_user", how="inner")
    .merge(dim_asignatura[["id_asignatura", "clave"]],
           left_on="materia_id", right_on="clave", how="inner")
    .merge(dim_tiempo[["id_periodo", "periodo_unam"]],
           left_on="periodo_norm", right_on="periodo_unam", how="inner")
    .assign(id_hecho=lambda d: range(1, len(d) + 1), cantidad=1)
    [["id_hecho", "id_estudiante", "id_asignatura", "id_periodo", "votacion_id", "fecha", "cantidad"]]
    .rename(columns={"id_estudiante": "id_dim_estudiante", "id_asignatura": "id_dim_asignatura",
                     "id_periodo": "id_dim_periodo", "votacion_id": "id_dim_votacion",
                     "fecha": "fecha_voto"})
)
_e = fact_extra_largo.merge(dim_tiempo, left_on="id_dim_periodo", right_on="id_periodo")
_e = _e.merge(dim_asignatura, left_on="id_dim_asignatura", right_on="id_asignatura")
rep_extras_por_materia = (
    _e.groupby(["periodo_unam", "clave", "nombre"], as_index=False)
    .agg(total_extras_largos=("cantidad", "sum"),
         alumnos_distintos=("id_dim_estudiante", "nunique"))
    .assign(ranking_mas_extras=lambda d: d["total_extras_largos"].rank(ascending=False, method="min").astype(int),
            ranking_menos_extras=lambda d: d["total_extras_largos"].rank(ascending=True, method="min").astype(int))
)

fact_inscripcion = (
    df_materias_inscritas
    .merge(dim_alumno[["id_estudiante", "id_auth_user"]],
           left_on="alumno_id", right_on="id_auth_user", how="inner")
    .merge(dim_asignatura[["id_asignatura", "clave"]],
           left_on="asignatura_id", right_on="clave", how="inner")
    .drop_duplicates(subset=["id_estudiante", "id_asignatura"])
    .assign(id_hecho=lambda d: range(1, len(d) + 1), cantidad=1)
    [["id_hecho", "id_estudiante", "id_asignatura", "cantidad"]]
    .rename(columns={"id_estudiante": "id_dim_estudiante", "id_asignatura": "id_dim_asignatura"})
)
_o = fact_inscripcion.merge(dim_asignatura, left_on="id_dim_asignatura", right_on="id_asignatura")
rep_optativas_mas_menos_elegidas = (
    _o[_o["es_optativa"] == 1]
    .groupby(["clave", "nombre"], as_index=False)
    .agg(total_inscripciones=("cantidad", "sum"),
         alumnos_distintos=("id_dim_estudiante", "nunique"))
    .assign(ranking_mas_elegidas=lambda d: d["total_inscripciones"].rank(ascending=False, method="min").astype(int),
            ranking_menos_elegidas=lambda d: d["total_inscripciones"].rank(ascending=True, method="min").astype(int))
    .sort_values("total_inscripciones", ascending=False)
)
print("fact_extra_largo:"); display(fact_extra_largo.head())
print("Materias con más/menos extra:"); display(rep_extras_por_materia.sort_values("total_extras_largos", ascending=False).head(10))
print(f"Optativas encontradas: {len(rep_optativas_mas_menos_elegidas)} (en tu versión: 0, tabla vacía)")
print("Optativas más elegidas:"); display(rep_optativas_mas_menos_elegidas.head(10))
print("Optativas menos elegidas:"); display(rep_optativas_mas_menos_elegidas.tail(10))

fact_extra_largo:


,id_hecho,id_dim_estudiante,id_dim_asignatura,id_dim_periodo,id_dim_votacion,fecha_voto,cantidad
0,1,8088,200,59,3,2010-01-03,1
1,2,8088,207,59,3,2010-01-03,1
2,3,8088,211,59,3,2010-01-03,1
3,4,7734,207,59,3,2010-01-04,1
4,5,8125,190,59,3,2010-01-03,1


Materias con más/menos extra:


,periodo_unam,clave,nombre,total_extras_largos,alumnos_distintos,ranking_mas_extras,ranking_menos_extras
219,20121,1206,ALGEBRA LINEAL,41,41,1,365
74,20111,1206,ALGEBRA LINEAL,40,40,2,364
220,20121,1207,CALCULO II,32,32,3,363
213,20121,1107,CALCULO I,30,30,4,362
75,20111,1207,CALCULO II,28,28,5,359
235,20121,1409,OPTIMIZACION LINEAL,28,28,5,359
47,20102,2502,ECUACIONES DIFERENCIALES,28,28,5,359
212,20121,1106,ALGEBRA SUPERIOR,27,27,8,358
222,20121,1209,METODOS NUMERICOS I,26,26,9,357
70,20111,1106,ALGEBRA SUPERIOR,24,24,10,355


Optativas encontradas: 26 (en tu versión: 0, tabla vacía)
Optativas más elegidas:


,clave,nombre,total_inscripciones,alumnos_distintos,ranking_mas_elegidas,ranking_menos_elegidas
2,1037,ANALISIS DE DECISIONES,85,85,1,26
3,1038,ANALISIS DE INVERSIONES,71,71,2,25
7,1043,PROGRAMACION LOGICA Y FUNCIONAL,70,70,3,23
12,1062,ADMINISTRACION DE REDES,70,70,3,23
9,1045,SISTEMAS OPERATIVOS,68,68,5,22
5,1040,INGENIERIA D SOFTWARE ORIENT.OBJETOS,62,62,6,21
1,1036,ANALISIS DE ALGORITMOS,59,59,7,20
15,1066,MINERIA DE DATOS,53,53,8,19
14,1065,EVALUACION DE PROYECTOS,49,49,9,18
8,1044,SIMULACION ESTOCASTICA,48,48,10,17


Optativas menos elegidas:


,clave,nombre,total_inscripciones,alumnos_distintos,ranking_mas_elegidas,ranking_menos_elegidas
10,1046,TEORIA DE LA COMPUTACION,30,30,16,10
16,1067,PROGRAMAC. PARALELA Y CONCURRENTE,20,20,18,8
20,1072,TEMAS SELECTOS DE MATEMATICAS APLIC.,20,20,18,8
22,1074,TEORIA DE LA CONFIABILIDAD,17,17,20,7
11,1047,VARIABLE COMPLEJA,16,16,21,6
17,1068,PRONOSTICOS II,9,9,22,5
6,1042,METODOS VARIACIONALES,8,8,23,4
23,2036,ADMINISTRACION,1,1,24,1
24,2045,SEMINARIO DE TITULACION,1,1,24,1
25,2048,TEMAS SELECTOS DE MATEMATICAS,1,1,24,1


## 9. Carga al data mart

Se crea `data_mart_pagmac` si no existe y se cargan dimensiones, hechos, reportes y `errores_etl`. `if_exists='replace'` es lo que ya usabas: cada corrida reescribe las tablas.

In [15]:
engine_srv = create_engine(f"mysql+pymysql://{USUARIO}:{PASSWORD}@{HOST}:{PUERTO}/")
with engine_srv.connect() as conn:
    conn.execute(text(f"CREATE DATABASE IF NOT EXISTS {BASE_DM} CHARACTER SET utf8mb4;"))
    conn.commit()
print("Base de datos 'data_mart_pagmac' creada o ya existente.")

engine_dm = create_engine(f"mysql+pymysql://{USUARIO}:{PASSWORD}@{HOST}:{PUERTO}/{BASE_DM}")


# to_sql(if_exists="replace") hace DROP TABLE, y MySQL no deja borrar una
# dimensión mientras un hecho tenga su FK (error 3730). Se borran primero
# los hechos (hijos) y luego las dimensiones (padres), con
# FOREIGN_KEY_CHECKS=0 por si alguna FK quedó de una carga anterior.
_tablas = ["fct_cal_alumno", "fct_extra_largo", "fct_inscripcion",
           "rep_metricas_materia", "rep_promedio_generacion",
           "rep_extras_por_materia", "rep_optativas_mas_menos_elegidas",
           "errores_etl", "dim_alumno", "dim_asignatura", "dim_tiempo", "dim_votacion"]
with engine_dm.connect() as conn:
    conn.execute(text("SET FOREIGN_KEY_CHECKS=0;"))
    for _t in _tablas:
        conn.execute(text(f"DROP TABLE IF EXISTS {_t};"))
    conn.execute(text("SET FOREIGN_KEY_CHECKS=1;"))
    conn.commit()
print("Tablas anteriores borradas (hechos primero, luego dimensiones).")

for _nombre, _df in [("dim_alumno", dim_alumno), ("dim_asignatura", dim_asignatura),
                     ("dim_tiempo", dim_tiempo), ("dim_votacion", dim_votacion),
                     ("fct_cal_alumno", fact_calificacion),
                     ("fct_extra_largo", fact_extra_largo),
                     ("fct_inscripcion", fact_inscripcion),
                     ("rep_metricas_materia", metricas_materia),
                     ("rep_promedio_generacion", promedio_generacion),
                     ("rep_extras_por_materia", rep_extras_por_materia),
                     ("rep_optativas_mas_menos_elegidas", rep_optativas_mas_menos_elegidas),
                     ("errores_etl", errores_etl)]:
    _df.to_sql(name=_nombre, con=engine_dm, if_exists="replace", index=False)
    print(f"{_nombre} cargada ({len(_df)} filas).")

Base de datos 'data_mart_pagmac' creada o ya existente.
Tablas anteriores borradas (hechos primero, luego dimensiones).
dim_alumno cargada (11234 filas).
dim_asignatura cargada (230 filas).
dim_tiempo cargada (63 filas).
dim_votacion cargada (5 filas).
fct_cal_alumno cargada (405157 filas).
fct_extra_largo cargada (2099 filas).
fct_inscripcion cargada (4491 filas).
rep_metricas_materia cargada (4903 filas).
rep_promedio_generacion cargada (17309 filas).
rep_extras_por_materia cargada (365 filas).
rep_optativas_mas_menos_elegidas cargada (26 filas).
errores_etl cargada (1 filas).


## 10. Validación post-carga (en el data mart, no borrando después)

Tu versión limpiaba los fantasmas DESPUÉS de cargar (celdas 30-31). Aquí ya no debe haber fantasmas: el ETL solo carga filas con llaves válidas. Si este conteo no da 0, no es para borrar en silencio: es para revisar el diagnóstico de llaves de la sección 2.

In [16]:
print(pd.read_sql("SHOW TABLES", engine_dm))
_dim_a = pd.read_sql("SELECT * FROM dim_alumno", engine_dm)
_dim_as = pd.read_sql("SELECT * FROM dim_asignatura", engine_dm)
_dim_t = pd.read_sql("SELECT * FROM dim_tiempo", engine_dm)
_fct = pd.read_sql("SELECT * FROM fct_cal_alumno", engine_dm)
print(f"dim_alumno: {_dim_a.shape} | dim_asignatura: {_dim_as.shape} | "
      f"dim_tiempo: {_dim_t.shape} | fct_cal_alumno: {_fct.shape}")
print("Duplicados en el grano del hecho (debe ser 0):",
      int(_fct.duplicated(subset=["id_dim_estudiante", "id_dim_asignatura", "id_dim_periodo"]).sum()))
print("Filas del hecho con alumno fantasma (debe ser 0):",
      int((~_fct["id_dim_estudiante"].isin(set(_dim_a["id_estudiante"]))).sum()))
print("Filas del hecho con asignatura fantasma (debe ser 0):",
      int((~_fct["id_dim_asignatura"].isin(set(_dim_as["id_asignatura"]))).sum()))
print("Filas del hecho con periodo fantasma (debe ser 0):",
      int((~_fct["id_dim_periodo"].isin(set(_dim_t["id_periodo"]))).sum()))
print("Filas en errores_etl:", len(pd.read_sql("SELECT * FROM errores_etl", engine_dm)))

          Tables_in_data_mart_pagmac
0                         dim_alumno
1                     dim_asignatura
2                         dim_tiempo
3                       dim_votacion
4                        errores_etl
5                     fct_cal_alumno
6                    fct_extra_largo
7                    fct_inscripcion
8             rep_extras_por_materia
9               rep_metricas_materia
10  rep_optativas_mas_menos_elegidas
11           rep_promedio_generacion
dim_alumno: (11234, 9) | dim_asignatura: (230, 8) | dim_tiempo: (63, 6) | fct_cal_alumno: (405157, 14)
Duplicados en el grano del hecho (debe ser 0): 0
Filas del hecho con alumno fantasma (debe ser 0): 0
Filas del hecho con asignatura fantasma (debe ser 0): 0
Filas del hecho con periodo fantasma (debe ser 0): 0
Filas en errores_etl: 1
